<h1> Instalasi Dependency & Tesseract Engine

In [3]:
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-ind
!pip install pytesseract opencv-python-headless

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,030 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,934 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/restricte

<h1>1. Pengerjaan alat OCR

In [1]:
import os
import cv2
import numpy as np

In [4]:
# Mengecek ketersediaan library OCR
try:
    import pytesseract
    HAS_PYTESSERACT = True
except ImportError:
    HAS_PYTESSERACT = False

<h3>    Melakukan serangkaian preprocessing pada gambar dokumen:

    1. Grayscale Conversion (Mengubah warna menjadi abu-abu)
    2. Denoising (Menghilangkan bintik/noise pada scan)
    3. Adaptive Thresholding / Binarization (Memperjelas kontras teks vs background)

In [5]:
def preprocess_image(image_input):
    # 1. Jika input berupa file path, baca gambarnya
    if isinstance(image_input, str):
        if not os.path.exists(image_input):
            raise FileNotFoundError(f"File gambar tidak ditemukan: {image_input}")
        image = cv2.imread(image_input)
    else:
        image = image_input

    # 2. Convert ke Grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 3. Hilangkan Noise (Denoising)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)

    # 4. Binarization dengan Otsu's Thresholding (Hitam-Putih Kontras Tinggi)
    _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return gray, thresh

In [6]:
def extract_text_from_image(image_path_or_array, lang='ind+eng'):
    """
    Fungsi utama untuk menjalankan preprocessing dan ekstraksi teks dengan Tesseract OCR.
    """
    # Jalankan preprocessing
    gray_img, preprocessed_img = preprocess_image(image_path_or_array)

    raw_text = ""

    if HAS_PYTESSERACT:
        try:
            # Konfigurasi Tesseract (PSM 6: Mengasumsikan blok teks tunggal yang seragam)
            custom_config = r'--oem 3 --psm 6'
            raw_text = pytesseract.image_to_string(preprocessed_img, lang=lang, config=custom_config)
        except Exception as e:
            raw_text = f"[ERROR OCR]: Gagal menjalankan Tesseract. Detail: {str(e)}\n" \
                       f"Pastikan engine Tesseract sudah terinstall di sistem."
    else:
        raw_text = "[WARNING]: Library pytesseract belum terinstall. Silakan install dengan 'pip install pytesseract'."

    return {
        "raw_text": raw_text.strip(),
        "preprocessed_image": preprocessed_img
    }

In [7]:
def create_sample_ktp_image(file_name="sample_ktp_dummy.png"):
    """
    Fungsi pembantu untuk membuat gambar KTP dummy sintetis menggunakan OpenCV.
    Berguna untuk pengujian awal di Colab/VS Code tanpa perlu upload file foto asli.
    """
    # Buat kanvas putih berukuran 600x400 piksel
    canvas = np.ones((400, 650, 3), dtype="uint8") * 255

    # Tambahkan header dan teks KTP dummy
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(canvas, "PROVINSI DKI JAKARTA", (180, 40), font, 0.7, (0, 0, 0), 2)
    cv2.putText(canvas, "JAKARTA PUSAT", (220, 70), font, 0.6, (0, 0, 0), 2)

    cv2.putText(canvas, "NIK             : 3171012304980001", (30, 120), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "Nama            : BUDI SANTOSO", (30, 150), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "Tempat/Tgl Lahir : JAKARTA, 23-04-1998", (30, 180), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "Jenis Kelamin   : LAKI-LAKI", (30, 210), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "Alamat          : JL MERDEKA NO 12", (30, 240), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "   RT/RW        : 001/002", (30, 270), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "   Kel/Desa     : GAMBIR", (30, 300), font, 0.5, (0, 0, 0), 1)
    cv2.putText(canvas, "Pendidikan      : SMK TEKNIK MESIN", (30, 330), font, 0.5, (0, 0, 0), 1)

    # Simpan gambar ke file
    cv2.imwrite(file_name, canvas)
    print(f"[INFO] Gambar KTP dummy berhasil dibuat: {file_name}")
    return file_name

In [8]:
if __name__ == "__main__":
    print("==================================================")
    print("       MODUL 1: OCR & PREPROCESSING PIPELINE     ")
    print("==================================================")

    # 1. Buat gambar sampel otomatis jika belum ada file
    dummy_ktp_path = "sample_ktp_dummy.png"
    create_sample_ktp_image(dummy_ktp_path)

    # 2. Jalankan pipeline OCR pada gambar sampel
    print("\n[PROCESS] Memproses gambar dan mengekstrak teks...")
    result = extract_text_from_image(dummy_ktp_path)

    # 3. Tampilkan Teks Mentah Hasil Ekstraksi
    print("\n--- HASIL EKSTRAKSI TEKS MENTAH (RAW OCR TEXT) ---")
    print(result["raw_text"])
    print("--------------------------------------------------")

       MODUL 1: OCR & PREPROCESSING PIPELINE     
[INFO] Gambar KTP dummy berhasil dibuat: sample_ktp_dummy.png

[PROCESS] Memproses gambar dan mengekstrak teks...

--- HASIL EKSTRAKSI TEKS MENTAH (RAW OCR TEXT) ---
PROVINSI DKI JAKARTA
JAKARTA PUSAT

NIK 1 3171012304980001
Nama : BUDI SANTOSO
Tempat/Tgl Lahir : JAKARTA, 23-04-1998
Jenis Kelamin» LAKI-LAKI
Alamat 2 JL MERDEKA NO 12

RT/RW : 001/002

Kel/Desa —— : GAMBIR
Pendidikan : SMK TEKNIK MESIN
--------------------------------------------------
